# 08 - Neighbourhood-Level Property Value Analysis: Sample Inspection

**Phase:** Exploratory Sample Inspection
**Notebook goal:** Identify which geography-related columns are available in the Property Tax Report and determine which one can support a future neighbourhood-level property value analysis.

---

## 2. Purpose

This notebook prepares the property-value side of the analysis for a future neighbourhood-level aggregation step. Before any cross-dataset comparison with building permit data can be attempted, we need to understand what geography fields are available in the Property Tax Report, how complete they are, and which one is most suitable for grouping properties by area.

This is a **read-only sample inspection**. No processed outputs are written. No aggregations are finalised. The full dataset is not loaded.

> **Important caveat:** BC Assessment assessed values are administrative property valuations used as the basis for property tax calculations. They are not MLS sale prices, transaction prices, or market values. Year-over-year changes in assessed value should not be interpreted as direct measures of market appreciation or depreciation. No causal claims are made between assessed value patterns and housing supply or permit activity.

## 3. Imports

We use `pathlib` for cross-platform path handling and `pandas` for tabular data inspection. No additional libraries are needed for this sample-inspection stage.

In [3]:
from pathlib import Path
import pandas as pd

## 4. Project Paths

All paths and configuration values are defined as named constants. `SAMPLE_SIZE` controls how many rows are loaded from the raw file. `CSV_SEPARATOR` is set to `";"` because the Property Tax Report uses semicolons as delimiters, as confirmed during initial data inspection in Notebook 01.

In [4]:
RAW_PROPERTY_TAX_PATH = Path('../data/raw/property_tax_report_raw.csv')
SAMPLE_SIZE           = 1000
CSV_SEPARATOR         = ';'

print(f'RAW_PROPERTY_TAX_PATH : {RAW_PROPERTY_TAX_PATH}')
print(f'SAMPLE_SIZE           : {SAMPLE_SIZE:,}')
print(f'CSV_SEPARATOR         : {repr(CSV_SEPARATOR)}')

RAW_PROPERTY_TAX_PATH : ..\data\raw\property_tax_report_raw.csv
SAMPLE_SIZE           : 1,000
CSV_SEPARATOR         : ';'


## 5. Safety Flag

`RUN_FULL_PROCESSING = False` prevents any future full-dataset logic from being triggered accidentally. This flag must remain `False` until the sample inspection logic in this notebook is reviewed and committed.

Do not set this to `True` unless a dedicated full-processing section has been planned and reviewed. The full Property Tax Report is approximately 443 MB and requires chunked reading.

In [5]:
RUN_FULL_PROCESSING = False

assert not RUN_FULL_PROCESSING, (
    'RUN_FULL_PROCESSING is True. This notebook is in sample-inspection mode. '
    'Review and commit the sample logic before enabling full processing.')

print(f'RUN_FULL_PROCESSING : {RUN_FULL_PROCESSING}')
print('Mode                : sample inspection only')
print(f'Rows to load        : {SAMPLE_SIZE:,}')

RUN_FULL_PROCESSING : False
Mode                : sample inspection only
Rows to load        : 1,000


## 6. Load Sample

We load only the first 1,000 rows of the raw Property Tax Report using `nrows=SAMPLE_SIZE`. The semicolon separator is set explicitly via `CSV_SEPARATOR`. No cleaning, filtering, or feature engineering is applied at this stage -- the goal is to see the raw column structure as it comes from the source file.

The full file is approximately 443 MB and is never loaded in this notebook.

In [6]:
df = pd.read_csv(RAW_PROPERTY_TAX_PATH, sep=CSV_SEPARATOR, nrows=SAMPLE_SIZE)

print(f'Rows loaded   : {len(df):,}')
print(f'Columns found : {len(df.columns):,}')

Rows loaded   : 1,000
Columns found : 30


## 7. Inspect Columns

We first print the full column list to see every field available in the dataset, then filter for columns that may relate to geography, area, or location. The filter uses a broad keyword search on column names -- this is intentionally inclusive at the inspection stage, since column naming conventions vary across datasets.

In [7]:
GEO_KEYWORDS = [
    'geo', 'area', 'neighbourhood', 'neighborhood', 'ward',
    'district', 'locality', 'coord', 'address', 'code',
    'zone', 'local', 'street', 'city', 'region', 'lat', 'lon', 'lng',]

print(f'Sample shape : {df.shape}')
print()
print('All columns:')
for col in df.columns:
    print(f'  {col}')
print()

geo_candidates = [
    col for col in df.columns
    if any(kw in col.lower() for kw in GEO_KEYWORDS)]

print(f'Geography candidate columns ({len(geo_candidates)} found):')
for col in geo_candidates:
    print(f'  {col}')

Sample shape : (1000, 30)

All columns:
  PID
  LEGAL_TYPE
  FOLIO
  LAND_COORDINATE
  ZONING_DISTRICT
  ZONING_CLASSIFICATION
  LOT
  PLAN
  BLOCK
  DISTRICT_LOT
  FROM_CIVIC_NUMBER
  TO_CIVIC_NUMBER
  STREET_NAME
  PROPERTY_POSTAL_CODE
  NARRATIVE_LEGAL_LINE1
  NARRATIVE_LEGAL_LINE2
  NARRATIVE_LEGAL_LINE3
  NARRATIVE_LEGAL_LINE4
  NARRATIVE_LEGAL_LINE5
  CURRENT_LAND_VALUE
  CURRENT_IMPROVEMENT_VALUE
  TAX_ASSESSMENT_YEAR
  PREVIOUS_LAND_VALUE
  PREVIOUS_IMPROVEMENT_VALUE
  YEAR_BUILT
  BIG_IMPROVEMENT_YEAR
  TAX_LEVY
  NEIGHBOURHOOD_CODE
  REPORT_YEAR
  NOTE

Geography candidate columns (6 found):
  LAND_COORDINATE
  ZONING_DISTRICT
  DISTRICT_LOT
  STREET_NAME
  PROPERTY_POSTAL_CODE
  NEIGHBOURHOOD_CODE


## 8. Missing-Rate Check

For each geography candidate column, we compute the non-null count, missing count, missing percentage, and number of unique values. Columns are sorted by missing percentage ascending so the most complete fields appear first. This tells us which fields are complete enough to support neighbourhood-level grouping across the full dataset.

In [8]:
rows = []
for col in geo_candidates:
    non_null = int(df[col].notna().sum())
    missing  = int(df[col].isna().sum())
    unique   = int(df[col].nunique(dropna=True))
    rows.append({
        'column':        col,
        'non_null':      non_null,
        'missing':       missing,
        'missing_pct':   round(missing / len(df) * 100, 2),
        'unique_values': unique,})

missing_df = pd.DataFrame(rows).sort_values('missing_pct')
display(missing_df.reset_index(drop=True))

,column,non_null,missing,missing_pct,unique_values
0,LAND_COORDINATE,1000,0,0.0,751
1,ZONING_DISTRICT,1000,0,0.0,158
2,STREET_NAME,1000,0,0.0,274
3,NEIGHBOURHOOD_CODE,1000,0,0.0,30
4,PROPERTY_POSTAL_CODE,984,16,1.6,724
5,DISTRICT_LOT,977,23,2.3,66


## 9. Value Preview

For each geography candidate column, we display the top 20 most frequent values including missing values. This reveals whether the field contains human-readable labels, numeric codes, or structured identifiers, and whether the values look consistent enough to support grouping across the full dataset.

In [9]:
TOP_N = 20

for col in geo_candidates:
    print(f'--- {col} ---')
    vc = df[col].value_counts(dropna=False).head(TOP_N)
    print(vc.to_string())
    print()

--- LAND_COORDINATE ---
LAND_COORDINATE
17258935    11
15660467     9
12060598     8
63818443     8
32071895     7
64017092     7
83431307     7
65419193     6
64003507     5
59012446     5
57919093     5
13058906     5
64017917     5
17259506     5
63817904     5
64018235     4
17559294     4
59012441     4
60111895     4
65526707     4

--- ZONING_DISTRICT ---
ZONING_DISTRICT
R1-1          241
RM-4           80
DD             41
C-3A           32
RT-8           31
C-2            30
R3-3           22
RT-7           20
C-2A           20
RM-3           16
CD-1 (450)     15
FCCDD          14
R5-3           14
IC-3           14
RM-3A          13
CD-1 (566)     12
RS-1           11
I-1            11
RT-5           10
RT-3            9

--- DISTRICT_LOT ---
DISTRICT_LOT
526     271
540     114
541      81
185      49
200A     49
THSL     40
264A     40
2027     28
FC       26
196      24
NaN      23
302      22
192      20
OGT      17
139      17
176      15
36       12
331      12
301     

## Value Preview Interpretation

The six candidate columns vary substantially in their suitability for a recruiter-readable neighbourhood-level analysis. The missing-rate check and value distributions together inform the field decision in Section 10.

**`LAND_COORDINATE`** is complete in the sample (0 missing, 751 unique values across 1,000 rows) but is too granular for a neighbourhood-level summary. Each value represents an individual land parcel, not a meaningful geographic group.

**`STREET_NAME`** is also complete (0 missing, 274 unique values) but is too granular for this stage. Street-level grouping would produce hundreds of narrow groups that are difficult to interpret and would not align with the neighbourhood-level boundaries used in building permit data.

**`PROPERTY_POSTAL_CODE`** has a small number of missing values (16 missing, 1.6%) and 724 unique values across 1,000 rows. Postal codes are too granular for neighbourhood-level grouping. The forward sortation area (first three characters) could reduce granularity, but this is a derived field that adds complexity without a clear benefit over using `NEIGHBOURHOOD_CODE` directly.

**`ZONING_DISTRICT`** is complete in the sample (0 missing, 158 unique values). However, zoning districts represent land-use designations, not residential neighbourhood geography. A property's zoning class is not the same as its neighbourhood, and this field is better suited for a zoning-level analysis than for the neighbourhood comparison intended here.

**`DISTRICT_LOT`** has the highest missing rate in the sample (23 missing, 2.3%) and 66 unique values. The field represents a legal land description unit used in the Torrens title system. It is less directly interpretable for a business audience than a neighbourhood label and introduces missing-value risk.

**`NEIGHBOURHOOD_CODE`** is the strongest candidate. It has no missing values in the sample, 30 unique values across 1,000 rows, and is intended specifically for neighbourhood-level classification of City of Vancouver properties. The number of categories is manageable for grouped aggregation and aligns with the neighbourhood-level scope of this analysis.

## 10. Preliminary Geography Field Decision

**Primary geography field candidate: `NEIGHBOURHOOD_CODE`**

`NEIGHBOURHOOD_CODE` will be used as the preliminary geography field for future neighbourhood-level aggregation. It is complete in the sample (0 missing values), less granular than address-level or coordinate fields, and is designed specifically for grouping City of Vancouver properties by neighbourhood.

Among the candidates identified in Section 7:

- `LAND_COORDINATE`, `STREET_NAME`, and `PROPERTY_POSTAL_CODE` are too granular for a neighbourhood-level summary.
- `ZONING_DISTRICT` reflects land-use classification, not neighbourhood geography.
- `DISTRICT_LOT` is a legal land description unit that is less interpretable for a business audience and has missing values in the sample.

> **Caveat:** `NEIGHBOURHOOD_CODE` is a coded field, not a human-readable neighbourhood name. A mapping table or official documentation may be needed later to translate codes into readable neighbourhood labels for recruiter-facing outputs.

_This decision is based on a 1,000-row sample. Missing rates and value distributions should be verified on the full dataset before committing to this field for full-dataset aggregation._

In [10]:
PRIMARY_GEOGRAPHY_FIELD = 'NEIGHBOURHOOD_CODE'

print(f'Primary geography field candidate: {PRIMARY_GEOGRAPHY_FIELD}')

Primary geography field candidate: NEIGHBOURHOOD_CODE


## 11. Future Output Placeholder

Once a geography field is selected and the full dataset is processed, a future notebook will produce:

**`data/processed/property_value_change_by_neighbourhood.csv`**

This file does not exist yet. It will be created after the geography field decision is made and reviewed.

Planned metrics (subject to revision after the field decision in Section 10):

| Column | Description |
|---|---|
| `neighbourhood` | Geography label for the group (value from the chosen column) |
| `property_count` | Number of property records in this neighbourhood group |
| `median_current_total_assessed_value` | Median total assessed value (current year) across properties in this group |
| `median_previous_total_assessed_value` | Median total assessed value (prior year) across properties in this group |
| `median_absolute_value_change` | Median year-over-year change in total assessed value |
| `median_percentage_value_change` | Median year-over-year percentage change in total assessed value |
| `mean_percentage_value_change` | Mean year-over-year percentage change (sensitive to outliers; use with caution) |
| `share_increase` | Share of properties in this group with a positive `percentage_value_change` |
| `share_decrease` | Share of properties in this group with a negative `percentage_value_change` |
| `share_missing_percentage_change` | Share of properties where `percentage_value_change` could not be computed |
| `extreme_high_count` | Number of properties with `percentage_value_change > 100` in this group |
| `extreme_low_count` | Number of properties with `percentage_value_change < -50` in this group |

> **Caveat:** All assessed value metrics in this output will reflect BC Assessment administrative valuations, not MLS sale prices or transaction prices. Neighbourhood-level summaries should be interpreted as administrative valuation patterns, not as direct measures of market appreciation or depreciation. No causal claims will be made between these patterns and housing supply or permit activity.

_This section will be updated once the geography field decision in Section 10 is finalised._

## 12. Sample Aggregation Dry Run

Before running the full 443 MB Property Tax Report through the chunked pipeline, this section tests the neighbourhood-level aggregation logic on the 1,000-row sample loaded in Section 6. The input columns, feature engineering formulas, output metrics, and validation checks are identical to those used in the full processing workflow in Section 13.

`VALUE_COLUMNS` and the extreme-change thresholds are defined at the top of the code cell below. The same constants are declared again in Section 13 alongside `PROCESSED_OUTPUT_PATH` and `CHUNKSIZE`, which are only relevant to the full processing path.

In [11]:
VALUE_COLUMNS = [
    'CURRENT_LAND_VALUE',
    'CURRENT_IMPROVEMENT_VALUE',
    'PREVIOUS_LAND_VALUE',
    'PREVIOUS_IMPROVEMENT_VALUE',]
EXTREME_HIGH_THRESHOLD = 50
EXTREME_LOW_THRESHOLD  = -50

sample_aggregation_df = df.copy()

for col in VALUE_COLUMNS:
    sample_aggregation_df[col] = pd.to_numeric(sample_aggregation_df[col], errors='coerce')

sample_aggregation_df['current_total_assessed_value'] = (
    sample_aggregation_df['CURRENT_LAND_VALUE'] + sample_aggregation_df['CURRENT_IMPROVEMENT_VALUE'])
sample_aggregation_df['previous_total_assessed_value'] = (
    sample_aggregation_df['PREVIOUS_LAND_VALUE'] + sample_aggregation_df['PREVIOUS_IMPROVEMENT_VALUE'])
sample_aggregation_df['absolute_value_change'] = (
    sample_aggregation_df['current_total_assessed_value'] - sample_aggregation_df['previous_total_assessed_value'])
valid_prev = sample_aggregation_df['previous_total_assessed_value'].where(
    sample_aggregation_df['previous_total_assessed_value'] > 0)
sample_aggregation_df['percentage_value_change'] = (
    sample_aggregation_df['absolute_value_change'] / valid_prev * 100)

def aggregate_neighbourhood_sample(group):
    pct   = group['percentage_value_change']
    count = len(group)
    valid = int(pct.notna().sum())
    inc   = int((pct > 0).sum())
    dec   = int((pct < 0).sum())
    nc    = int((pct == 0).sum())
    miss  = int(pct.isna().sum())
    ext_h = int((pct > EXTREME_HIGH_THRESHOLD).sum())
    ext_l = int((pct < EXTREME_LOW_THRESHOLD).sum())
    return pd.Series({
        'property_count':                      count,
        'valid_percentage_change_count':       valid,
        'median_current_total_assessed_value':  group['current_total_assessed_value'].median(),
        'median_previous_total_assessed_value': group['previous_total_assessed_value'].median(),
        'median_absolute_value_change':         group['absolute_value_change'].median(),
        'median_percentage_value_change':       pct.median(),
        'mean_percentage_value_change':         pct.mean(),
        'increase_count':                      inc,
        'decrease_count':                      dec,
        'no_change_count':                     nc,
        'missing_percentage_change_count':     miss,
        'share_increase':                      inc  / count * 100,
        'share_decrease':                      dec  / count * 100,
        'share_no_change':                     nc   / count * 100,
        'share_missing_percentage_change':     miss / count * 100,
        'extreme_high_count':                  ext_h,
        'extreme_low_count':                   ext_l,})

sample_neighbourhood_df = (
    sample_aggregation_df
    .groupby(PRIMARY_GEOGRAPHY_FIELD, dropna=False)
    .apply(aggregate_neighbourhood_sample)
    .reset_index()
    .rename(columns={PRIMARY_GEOGRAPHY_FIELD: 'neighbourhood_code'}))

sample_row_count = len(sample_aggregation_df)

agg_count = sample_neighbourhood_df['property_count'].sum()
assert agg_count == sample_row_count, (
    f'property_count sum mismatch: {agg_count:,} != {sample_row_count:,}')
print(f'Row count check           : {agg_count:,} rows -- OK')

valid_plus_missing = (
    sample_neighbourhood_df['valid_percentage_change_count'].sum()
    + sample_neighbourhood_df['missing_percentage_change_count'].sum())
assert valid_plus_missing == sample_row_count, (
    f'valid + missing ({valid_plus_missing:,}) != sample rows ({sample_row_count:,})')
print('valid + missing check     : OK')

assert sample_neighbourhood_df['neighbourhood_code'].nunique() == len(sample_neighbourhood_df), (
    'Duplicate neighbourhood codes found')
print(f'Neighbourhood codes found : {len(sample_neighbourhood_df):,}')
print('No duplicate codes        : OK')
print()
display(
    sample_neighbourhood_df
    .sort_values('property_count', ascending=False)
    .reset_index(drop=True))

Row count check           : 1,000.0 rows -- OK
valid + missing check     : OK
Neighbourhood codes found : 30
No duplicate codes        : OK



,neighbourhood_code,property_count,valid_percentage_change_count,median_current_total_assessed_value,median_previous_total_assessed_value,median_absolute_value_change,median_percentage_value_change,mean_percentage_value_change,increase_count,decrease_count,no_change_count,missing_percentage_change_count,share_increase,share_decrease,share_no_change,share_missing_percentage_change,extreme_high_count,extreme_low_count
0,2,210.0,210.0,1416500.0,1341000.0,32805.0,2.030866,2.520905,139.0,67.0,4.0,0.0,66.190476,31.904762,1.904762,0.000000,0.0,0.0
1,13,143.0,140.0,900500.0,883500.0,11000.0,1.365763,-1.435750,74.0,66.0,0.0,3.0,51.748252,46.153846,0.000000,2.097902,0.0,2.0
2,26,104.0,101.0,729000.0,719000.0,12000.0,1.890990,0.430952,62.0,38.0,1.0,3.0,59.615385,36.538462,0.961538,2.884615,0.0,0.0
3,7,75.0,73.0,1027000.0,971000.0,33000.0,5.973223,5.200388,55.0,18.0,0.0,2.0,73.333333,24.000000,0.000000,2.666667,0.0,0.0
4,1,65.0,65.0,2655000.0,2659000.0,23000.0,1.574539,2.910657,39.0,26.0,0.0,0.0,60.000000,40.000000,0.000000,0.000000,0.0,0.0
5,10,37.0,36.0,3950000.0,3919500.0,18500.0,0.627324,2.205251,21.0,15.0,0.0,1.0,56.756757,40.540541,0.000000,2.702703,0.0,0.0
6,5,31.0,31.0,3313400.0,3339300.0,17000.0,0.656400,2.873754,18.0,13.0,0.0,0.0,58.064516,41.935484,0.000000,0.000000,0.0,0.0
7,3,31.0,31.0,3048600.0,2984200.0,-39000.0,-1.395520,-1.423717,6.0,25.0,0.0,0.0,19.354839,80.645161,0.000000,0.000000,0.0,0.0
8,29,29.0,28.0,806000.0,753500.0,22000.0,2.528797,0.250362,18.0,9.0,1.0,1.0,62.068966,31.034483,3.448276,3.448276,0.0,0.0
9,14,28.0,23.0,1416000.0,1479000.0,76780.0,6.703911,5.945930,19.0,4.0,0.0,5.0,67.857143,14.285714,0.000000,17.857143,0.0,0.0


> **Dry run note:** This output is based on the 1,000-row development sample. It is not a representative neighbourhood-level analysis and should not be interpreted as a project conclusion. The purpose is to validate the aggregation logic, output column structure, and assertion checks before applying them to the full dataset.

## 13. Full Processing Design

This section defines the chunked processing workflow for generating neighbourhood-level property value metrics from the full Property Tax Report. The geography field confirmed in Section 10 -- `NEIGHBOURHOOD_CODE` -- is used to group all property records.

The code in this section will not execute until `RUN_FULL_PROCESSING` is explicitly set to `True`. No CSV file is exported here. This is a design scaffold to be reviewed before full processing is triggered.

The following constants define the output file path, chunk size, geography field, value columns, and extreme-change thresholds for the full processing workflow.

In [12]:
PROCESSED_OUTPUT_PATH   = Path('../data/processed/property_value_change_by_neighbourhood.csv')
CHUNKSIZE               = 100_000
PRIMARY_GEOGRAPHY_FIELD = 'NEIGHBOURHOOD_CODE'

VALUE_COLUMNS = [
    'CURRENT_LAND_VALUE',
    'CURRENT_IMPROVEMENT_VALUE',
    'PREVIOUS_LAND_VALUE',
    'PREVIOUS_IMPROVEMENT_VALUE',]

EXTREME_HIGH_THRESHOLD = 50
EXTREME_LOW_THRESHOLD  = -50

print(f'PROCESSED_OUTPUT_PATH   : {PROCESSED_OUTPUT_PATH}')
print(f'CHUNKSIZE               : {CHUNKSIZE:,}')
print(f'PRIMARY_GEOGRAPHY_FIELD : {PRIMARY_GEOGRAPHY_FIELD}')
print(f'EXTREME_HIGH_THRESHOLD  : {EXTREME_HIGH_THRESHOLD}')
print(f'EXTREME_LOW_THRESHOLD   : {EXTREME_LOW_THRESHOLD}')

PROCESSED_OUTPUT_PATH   : ..\data\processed\property_value_change_by_neighbourhood.csv
CHUNKSIZE               : 100,000
PRIMARY_GEOGRAPHY_FIELD : NEIGHBOURHOOD_CODE
EXTREME_HIGH_THRESHOLD  : 50
EXTREME_LOW_THRESHOLD   : -50


## Feature Engineering Definitions

The four derived value-change columns are computed using the same logic as Notebook 05's `compute_value_change_features()`. This consistency is required because `property_value_change_summary.csv` was already generated from the full dataset using those definitions -- the neighbourhood-level aggregation must use identical formulas.

| Derived column | Formula | Null when |
|---|---|---|
| `current_total_assessed_value` | `CURRENT_LAND_VALUE + CURRENT_IMPROVEMENT_VALUE` | Either component is missing |
| `previous_total_assessed_value` | `PREVIOUS_LAND_VALUE + PREVIOUS_IMPROVEMENT_VALUE` | Either component is missing |
| `absolute_value_change` | `current_total_assessed_value - previous_total_assessed_value` | Either total is null |
| `percentage_value_change` | `absolute_value_change / previous_total_assessed_value * 100` | Previous total is null, zero, or negative |

Assessed values are administrative property valuations. Year-over-year changes in assessed value are not measures of market price appreciation or depreciation.

In [13]:
if RUN_FULL_PROCESSING:

    def compute_value_change_features(chunk):
        """Matches Notebook 05 compute_value_change_features."""
        for col in VALUE_COLUMNS:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce')
        chunk['current_total_assessed_value'] = (
            chunk['CURRENT_LAND_VALUE'] + chunk['CURRENT_IMPROVEMENT_VALUE'])
        chunk['previous_total_assessed_value'] = (
            chunk['PREVIOUS_LAND_VALUE'] + chunk['PREVIOUS_IMPROVEMENT_VALUE'])
        chunk['absolute_value_change'] = (
            chunk['current_total_assessed_value'] - chunk['previous_total_assessed_value'])
        valid_prev = chunk['previous_total_assessed_value'].where(
            chunk['previous_total_assessed_value'] > 0)
        chunk['percentage_value_change'] = (
            chunk['absolute_value_change'] / valid_prev * 100)
        return chunk

    SLIM_COLUMNS = [
        PRIMARY_GEOGRAPHY_FIELD,
        'current_total_assessed_value',
        'previous_total_assessed_value',
        'absolute_value_change',
        'percentage_value_change',]

    slim_chunks          = []
    total_rows_processed = 0

    for chunk in pd.read_csv(
        RAW_PROPERTY_TAX_PATH,
        sep=CSV_SEPARATOR,
        usecols=[PRIMARY_GEOGRAPHY_FIELD] + VALUE_COLUMNS,
        chunksize=CHUNKSIZE,):
        chunk = compute_value_change_features(chunk)
        slim_chunks.append(chunk[SLIM_COLUMNS].copy())
        total_rows_processed += len(chunk)

    full_df = pd.concat(slim_chunks, ignore_index=True)
    print(f'Total rows processed          : {total_rows_processed:,}')
    print(f'Concatenated shape            : {full_df.shape}')
    print()

    def aggregate_neighbourhood(group):
        pct   = group['percentage_value_change']
        count = len(group)
        valid = int(pct.notna().sum())
        inc   = int((pct > 0).sum())
        dec   = int((pct < 0).sum())
        nc    = int((pct == 0).sum())
        miss  = int(pct.isna().sum())
        ext_h = int((pct > EXTREME_HIGH_THRESHOLD).sum())
        ext_l = int((pct < EXTREME_LOW_THRESHOLD).sum())
        return pd.Series({
            'property_count':                      count,
            'valid_percentage_change_count':       valid,
            'median_current_total_assessed_value':  group['current_total_assessed_value'].median(),
            'median_previous_total_assessed_value': group['previous_total_assessed_value'].median(),
            'median_absolute_value_change':         group['absolute_value_change'].median(),
            'median_percentage_value_change':       pct.median(),
            'mean_percentage_value_change':         pct.mean(),
            'increase_count':                      inc,
            'decrease_count':                      dec,
            'no_change_count':                     nc,
            'missing_percentage_change_count':     miss,
            'share_increase':                      inc  / count * 100,
            'share_decrease':                      dec  / count * 100,
            'share_no_change':                     nc   / count * 100,
            'share_missing_percentage_change':     miss / count * 100,
            'extreme_high_count':                  ext_h,
            'extreme_low_count':                   ext_l,})

    neighbourhood_summary = (
        full_df
        .groupby(PRIMARY_GEOGRAPHY_FIELD, dropna=False)
        .apply(aggregate_neighbourhood)
        .reset_index()
        .rename(columns={PRIMARY_GEOGRAPHY_FIELD: 'neighbourhood_code'}))

    total_agg_count = neighbourhood_summary['property_count'].sum()
    assert total_agg_count == total_rows_processed, (
        f'Row count mismatch: aggregated {total_agg_count:,} != processed {total_rows_processed:,}')
    print(f'Aggregated property_count sum : {total_agg_count:,}')
    print(f'Row count check               : OK')

    valid_plus_missing = (
        neighbourhood_summary['valid_percentage_change_count'].sum()
        + neighbourhood_summary['missing_percentage_change_count'].sum())
    assert valid_plus_missing == total_rows_processed, (
        f'valid + missing ({valid_plus_missing:,}) != total rows ({total_rows_processed:,})')
    print('valid + missing check         : OK')

    assert neighbourhood_summary['neighbourhood_code'].nunique() == len(neighbourhood_summary), (
        'Duplicate neighbourhood codes found in output')
    print(f'Neighbourhood codes found     : {len(neighbourhood_summary):,}')
    print('No duplicate codes            : OK')
    print()
    display(neighbourhood_summary.head())

    neighbourhood_summary.to_csv(PROCESSED_OUTPUT_PATH, index=False)
    print(f'Saved output to: {PROCESSED_OUTPUT_PATH}')

else:
    print('Full processing skipped. Set RUN_FULL_PROCESSING = True to run.')

Full processing skipped. Set RUN_FULL_PROCESSING = True to run.


## Processing Caveats

`NEIGHBOURHOOD_CODE` is a coded field, not a human-readable neighbourhood name. A mapping table or official documentation may be needed later to translate codes into readable neighbourhood labels before this output is suitable for recruiter-facing or public-facing presentations.

All assessed value metrics in the planned output reflect BC Assessment administrative property valuations used for property tax purposes. They are not MLS sale prices, transaction prices, or direct measures of market appreciation. No causal claims will be made between neighbourhood-level assessed value patterns and housing supply, building permit activity, or market prices.